## Práctica: word2vec / skip-gram

In [1]:
# Leemos el corpus: una frase por línea, descartando líneas vacías
with open("dataset_word2vec.txt", encoding="utf-8") as f:
    frases = [linea.strip() for linea in f if linea.strip()]

print(len(frases), "frases")
print(frases[:3])

251 frases
['el perro corre en el parque', 'el perro come carne', 'el perro bebe agua']


Entrenar un modelo el cual aprenda nuestro vocabulario, para crear un diccionario el cual tenga la palabra y su id como valor dentro de un diccionario, después crear el mismo diccionario inverso para poder recuperar las palabras

In [2]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

# Tokenizador a nivel de palabra: cada palabra del vocabulario recibe un id,
# y las palabras desconocidas se mapean a <unk>
tokenizer = Tokenizer(models.WordLevel(unk_token="<unk>"))
tokenizer.normalizer = normalizers.Lowercase()          # todo a minúsculas
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()   # separar por espacios / puntuación

In [3]:
# Construimos el vocabulario a partir de las frases del corpus
trainer = trainers.WordLevelTrainer(special_tokens=["<unk>"])
tokenizer.train_from_iterator(frases, trainer=trainer)

In [4]:
vocab_size = tokenizer.get_vocab_size()
print("Tamaño del vocabulario:", vocab_size)

# voc_dicc: palabra -> id ; voc_dicc_inverso: id -> palabra (para recuperar palabras a partir de ids)
voc_dicc = tokenizer.get_vocab()
voc_dicc_inverso = {i: w for w, i in voc_dicc.items()}
print(list(voc_dicc.items())[:10])
print(list(voc_dicc_inverso.items())[:10])

Tamaño del vocabulario: 357
[('termina', 171), ('suena', 169), ('queso', 156), ('corta', 224), ('encendido', 237), ('cacao', 205), ('bombero', 200), ('húmeda', 263), ('camarero', 209), ('<unk>', 0)]
[(171, 'termina'), (169, 'suena'), (156, 'queso'), (224, 'corta'), (237, 'encendido'), (205, 'cacao'), (200, 'bombero'), (263, 'húmeda'), (209, 'camarero'), (0, '<unk>')]


In [5]:
# Convertimos cada frase en su lista de ids
frases_ids = [tokenizer.encode(f).ids for f in frases]
print(frases[0], "->", frases_ids[0])

el perro corre en el parque -> [2, 13, 22, 5, 2, 35]


In [6]:
def generar_pares(frases_ids, ventana):
    """Genera pares (centro, contexto): para cada palabra central, todas las
    palabras a una distancia <= ventana dentro de la misma frase."""
    pares = []
    for ids in frases_ids:
        for i, centro in enumerate(ids):
            for j in range(max(0, i - ventana), min(len(ids), i + ventana + 1)):
                if j != i:  # la palabra no es contexto de sí misma
                    pares.append((centro, ids[j]))
    return pares

ventana = 2
pares = generar_pares(frases_ids, ventana)
print(len(pares), "pares")
print([(voc_dicc_inverso[c], voc_dicc_inverso[x]) for c, x in pares[:5]])

3086 pares
[('el', 'perro'), ('el', 'corre'), ('perro', 'el'), ('perro', 'corre'), ('perro', 'en')]


## Modelo Skip-gram

La red tiene dos capas:
- `emb` (`nn.Embedding`): tabla de tamaño `vocab_size x dim`. La fila *i* es el vector (embedding) de la palabra *i*. **Es lo que queremos aprender.**
- `out` (`nn.Linear`): proyecta el embedding a una puntuación por cada palabra del vocabulario.

Dada la palabra central, el modelo predice una distribución (softmax) sobre las posibles palabras de contexto y se entrena con *cross-entropy* contra la palabra de contexto real. Palabras que aparecen en contextos parecidos acaban con embeddings parecidos.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd

class SkipGram(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim)  
        self.out = nn.Linear(dim, vocab_size)

    def forward(self, centro):
        return self.out(self.emb(centro))

## Entrenamiento

Encapsulamos el entrenamiento en una función para poder repetirlo luego con distintos hiperparámetros (dimensión, ventana, épocas, learning rate).

In [8]:
def entrenar(dim=20, ventana=2, epocas=100, lr=0.01, batch_size=64, seed=0, verbose=False):
    torch.manual_seed(seed)  # reproducibilidad

    # Pares (centro, contexto) con la ventana indicada -> DataLoader
    pares = generar_pares(frases_ids, ventana)
    centros = torch.tensor([p[0] for p in pares])
    contextos = torch.tensor([p[1] for p in pares])
    loader = DataLoader(TensorDataset(centros, contextos), batch_size=batch_size, shuffle=True)

    modelo = SkipGram(vocab_size, dim)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    criterio = nn.CrossEntropyLoss()  # incluye el softmax

    historial = []
    for epoca in range(epocas):
        total = 0.0
        for c, x in loader:
            optimizador.zero_grad()
            perdida = criterio(modelo(c), x)
            perdida.backward()
            optimizador.step()
            total += perdida.item() * len(c)
        historial.append(total / len(pares))  # pérdida media de la época
        if verbose and (epoca + 1) % 20 == 0:
            print(f"Época {epoca + 1}: pérdida = {historial[-1]:.4f}")
    return modelo, historial

In [9]:
# Modelo base
modelo, historial = entrenar(dim=20, ventana=2, epocas=100, lr=0.01, verbose=True)

Época 20: pérdida = 2.9138


Época 40: pérdida = 2.7978


Época 60: pérdida = 2.7707


Época 80: pérdida = 2.7593


Época 100: pérdida = 2.7491


## Vecinos más cercanos

Usamos la **similitud coseno** entre embeddings: normalizamos todos los vectores a norma 1 y el producto escalar da directamente el coseno.

In [10]:
def embeddings_normalizados(modelo):
    # Matriz (vocab_size, dim) con cada fila de norma 1
    return F.normalize(modelo.emb.weight.detach(), dim=1)

def vecinos(modelo, palabra, k=5):
    E = embeddings_normalizados(modelo)
    sims = E @ E[voc_dicc[palabra]]      # coseno con todas las palabras
    sims[voc_dicc[palabra]] = -1         # excluimos la propia palabra
    sims[voc_dicc["<unk>"]] = -1         # y el token <unk> (nunca aparece, su vector es aleatorio)
    valores, ids = sims.topk(k)
    return [(voc_dicc_inverso[i.item()], round(v.item(), 3)) for v, i in zip(valores, ids)]

palabras = ["perro", "gata", "coche", "parís", "francia"]
tabla_vecinos = pd.DataFrame(
    {p: [f"{w} ({s})" for w, s in vecinos(modelo, p, k=5)] for p in palabras},
    index=[f"vecino {i}" for i in range(1, 6)],
).T
tabla_vecinos

,vecino 1,vecino 2,vecino 3,vecino 4,vecino 5
perro,gato (0.819),vaso (0.518),ratón (0.508),cuaderno (0.507),ingeniero (0.507)
gata,perra (0.766),cuerda (0.62),fuente (0.587),gato (0.576),alumna (0.567)
coche,me (0.606),campo (0.568),conductora (0.561),fruta (0.558),chocolate (0.544)
parís,roma (0.878),madrid (0.771),españa (0.702),francia (0.653),ciudad (0.63)
francia,italia (0.892),españa (0.807),madrid (0.713),roma (0.669),vive (0.666)


## Analogías

Para resolver `a : b :: c : ?` calculamos `b - a + c` y buscamos la palabra más cercana (por coseno), excluyendo `a`, `b` y `c`.
Por ejemplo `parís : francia :: madrid : ?` → `francia - parís + madrid ≈ españa`.

In [11]:
def analogia(modelo, a, b, c, k=3):
    E = embeddings_normalizados(modelo)
    consulta = F.normalize(E[voc_dicc[b]] - E[voc_dicc[a]] + E[voc_dicc[c]], dim=0)
    sims = E @ consulta
    for w in (a, b, c, "<unk>"):  # la respuesta no puede ser una palabra de la pregunta ni <unk>
        sims[voc_dicc[w]] = -1
    valores, ids = sims.topk(k)
    return [(voc_dicc_inverso[i.item()], round(v.item(), 3)) for v, i in zip(valores, ids)]

# (a, b, c, respuesta esperada)
analogias = [
    ("parís", "francia", "madrid", "españa"),
    ("madrid", "españa", "roma", "italia"),
    ("roma", "italia", "parís", "francia"),
    ("perro", "perra", "gato", "gata"),
    ("gato", "gata", "perro", "perra"),
    ("rey", "reina", "el", "la"),
]

def tabla_analogias(modelo):
    filas = []
    for a, b, c, esperada in analogias:
        respuesta = analogia(modelo, a, b, c, k=1)[0][0]
        filas.append({"analogía": f"{a} : {b} :: {c} : ?", "esperada": esperada, "respuesta": respuesta})
    return pd.DataFrame(filas)

tabla_analogias(modelo)

,analogía,esperada,respuesta
0,parís : francia :: madrid : ?,españa,italia
1,madrid : españa :: roma : ?,italia,italia
2,roma : italia :: parís : ?,francia,españa
3,perro : perra :: gato : ?,gata,gata
4,gato : gata :: perro : ?,perra,perra
5,rey : reina :: el : ?,la,caluroso


## Influencia de los hiperparámetros

Variamos un hiperparámetro cada vez respecto al modelo base (`dim=20, ventana=2, epocas=100, lr=0.01`) y anotamos la pérdida final y cuántas de las 6 analogías acierta.

In [12]:
base = dict(dim=20, ventana=2, epocas=100, lr=0.01)
experimentos = [
    {},                                   # modelo base
    dict(dim=5), dict(dim=50), dict(dim=100),
    dict(ventana=1), dict(ventana=4),
    dict(epocas=20), dict(epocas=300),
    dict(lr=0.001), dict(lr=0.05),
]

resultados = []
for cambio in experimentos:
    m, h = entrenar(**{**base, **cambio})
    t = tabla_analogias(m)
    resultados.append({
        "cambio": ", ".join(f"{k}={v}" for k, v in cambio.items()) or "base",
        "pérdida final": round(h[-1], 3),
        "analogías acertadas": int((t["respuesta"] == t["esperada"]).sum()),
    })

pd.DataFrame(resultados)

,cambio,pérdida final,analogías acertadas
0,base,2.749,3
1,dim=5,3.333,5
2,dim=50,2.827,2
3,dim=100,2.902,3
4,ventana=1,2.193,2
5,ventana=4,3.057,2
6,epocas=20,2.914,2
7,epocas=300,2.723,3
8,lr=0.001,3.107,1
9,lr=0.05,3.271,2


## Comentario sobre los hiperparámetros

- **Dimensión (`dim`)**: es el que más influye. Con un corpus tan pequeño, dimensiones grandes (50–100) permiten memorizar en vez de generalizar y las analogías empeoran; con `dim=5` el modelo se ve obligado a agrupar las palabras con contextos parecidos y acierta 5/6.
- **Ventana**: define qué cuenta como contexto. Con `ventana=1` solo se ven artículos y preposiciones vecinas (menor pérdida, pero menos información semántica); con `ventana=4` entra más ruido. Una ventana intermedia (2) funciona mejor aquí.
- **Learning rate y épocas**: deciden si el modelo llega a converger. Con `lr=0.001` o 20 épocas el entrenamiento se queda corto (1–2 aciertos); con `lr=0.05` es inestable (la pérdida final empeora).

Una pérdida más baja no implica mejores embeddings (`ventana=1` tiene la menor pérdida y no es la mejor). Además, con tan pocos datos las diferencias de 1 analogía pueden deberse al azar de la inicialización.